<a href="https://colab.research.google.com/github/devmurarijay13/pytorch-deep-learning/blob/main/pytorch_training_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.sample(5)

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
410,905502,B,11.36,17.57,72.49,399.8,0.08858,0.05313,0.02783,0.02100,...,36.32,85.07,521.3,0.1453,0.1622,0.1811,0.08698,0.2973,0.07745,NaN
227,88147102,B,15.00,15.51,97.45,684.5,0.08371,0.10960,0.06505,0.03780,...,19.31,114.20,808.2,0.1136,0.3627,0.3402,0.13790,0.2954,0.08362,NaN
127,866203,M,19.00,18.91,123.40,1138.0,0.08217,0.08028,0.09271,0.05627,...,25.73,148.20,1538.0,0.1021,0.2264,0.3207,0.12180,0.2841,0.06541,NaN
65,859283,M,14.78,23.94,97.40,668.3,0.11720,0.14790,0.12670,0.09029,...,33.39,114.60,925.1,0.1648,0.3416,0.3024,0.16140,0.3321,0.08911,NaN
215,8810987,M,13.86,16.93,90.96,578.9,0.10260,0.15170,0.09901,0.05602,...,26.93,104.40,750.1,0.1460,0.4370,0.4636,0.16540,0.3630,0.10590,NaN


In [ ]:
df.drop(columns=['id','Unnamed: 32'],inplace=True)

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns='diagnosis'),df['diagnosis'],random_state=42,test_size=0.25)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [ ]:
#### np arr to pytorch tensor
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [ ]:
X_train_tensor.shape

torch.Size([426, 30])

In [ ]:
class MySimpleNN:
    def __init__(self,X):
        self.weights = torch.rand(X.shape[1],1,dtype=torch.float64,requires_grad=True)
        self.bias = torch.zeros(1,dtype=torch.float64,requires_grad=True)

    def forward(self,X):
        z = torch.matmul(X,self.weights) + self.bias
        y_pred = torch.sigmoid(z)
        return y_pred

    def loss_func(self,y_pred,y):
        epsilon = 1e-7
        y_pred = torch.clamp(y_pred,epsilon,1-epsilon)

        ##calculate loss
        loss = -(y * torch.log(y_pred) + (1-y) * torch.log(1-y_pred)).mean()
        return loss

In [ ]:
#### imp parameters
learning_rate = 0.1
epochs = 25

In [ ]:
#### training pipeline
model = MySimpleNN(X_train_tensor)

### define loop
for epoch in range(epochs):

    ### forward pass
    y_pred = model.forward(X_train_tensor)
    # print(y_pred)

    ### loss calculate
    loss = model.loss_func(y_pred,y_train_tensor)
    # print(f"epoch : {epoch + 1}, loss : {loss.item()}")

    #### backward pass
    loss.backward()

    #### parameters update
    with torch.no_grad(): ###-------don't track gradients in this process
        model.weights -= learning_rate * model.weights.grad
        model.bias -= learning_rate * model.bias.grad

    #### zero gradients
    model.weights.grad.zero_()
    model.bias.grad.zero_()

    print(f"epoch : {epoch+1} loss : {loss.item()}")

epoch : 1 loss : 3.9053013877987297
epoch : 2 loss : 3.7934731070699645
epoch : 3 loss : 3.677189623892949
epoch : 4 loss : 3.5544825169439576
epoch : 5 loss : 3.427423768271413
epoch : 6 loss : 3.296618624649378
epoch : 7 loss : 3.155796180161309
epoch : 8 loss : 3.010887745630404
epoch : 9 loss : 2.8652476278707852
epoch : 10 loss : 2.716827198155758
epoch : 11 loss : 2.5688525689852297
epoch : 12 loss : 2.417899531067779
epoch : 13 loss : 2.263083228734064
epoch : 14 loss : 2.107392330884927
epoch : 15 loss : 1.954338784622989
epoch : 16 loss : 1.8066413444502707
epoch : 17 loss : 1.6617780531489033
epoch : 18 loss : 1.5277424164989493
epoch : 19 loss : 1.405850929158947
epoch : 20 loss : 1.2936454216116062
epoch : 21 loss : 1.1966204989893128
epoch : 22 loss : 1.1151352780781092
epoch : 23 loss : 1.0486785828293375
epoch : 24 loss : 0.9958108841292461
epoch : 25 loss : 0.9543939350448155


In [ ]:
model.weights

tensor([[ 0.2745],
        [-0.0111],
        [-0.1539],
        [-0.3688],
        [ 0.4115],
        [-0.3540],
        [-0.5991],
        [ 0.0417],
        [-0.0308],
        [-0.1909],
        [ 0.1956],
        [ 0.6528],
        [-0.2090],
        [ 0.2875],
        [ 0.5903],
        [ 0.3663],
        [ 0.1771],
        [-0.1594],
        [ 0.0825],
        [ 0.4442],
        [ 0.0237],
        [ 0.2341],
        [ 0.4802],
        [ 0.2917],
        [ 0.4368],
        [-0.0653],
        [-0.2843],
        [-0.0803],
        [-0.0554],
        [ 0.3507]], dtype=torch.float64, requires_grad=True)

In [ ]:
model.bias

tensor([-0.1369], dtype=torch.float64, requires_grad=True)

In [ ]:
with torch.no_grad():
    y_pred = model.forward(X_test_tensor)
    y_pred = (y_pred > 0.5).float()
    accuracy = (y_pred == y_test_tensor).float().mean()
    print(f"Accuracy : {accuracy.item()}")

Accuracy : 0.5111252665519714
